# ROB Tables - Quick Inspection (processed_export)

This notebook loads the latest `processed_export_*.json`, builds `df_tables`, and lets you print each table with `show_table(index)`.

Each table entry keeps the markdown and PDF paths close at hand for quick manual verification.

In [3]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

EXPORTS_DIR = Path('../paper_pipeline_data/exports')
processed_files = sorted(EXPORTS_DIR.glob('processed_export_*.json'))
if not processed_files:
    raise FileNotFoundError('No processed_export_*.json found in paper_pipeline_data/exports')

proc_file = processed_files[-1]
print('Using:', proc_file)
with open(proc_file, 'r', encoding='utf-8') as f:
    processed = json.load(f)

print(f'Loaded {len(processed)} documents')

Using: ../paper_pipeline_data/exports/processed_export_1778938206.json
Loaded 134 documents


In [4]:
rows = []

for doc in processed:
    doi = doc.get('paper_id') or doc.get('pmcid') or doc.get('paper') or doc.get('doi')
    md_path = doc.get('md') or doc.get('md_path')
    pdf_path = doc.get('pdf_path')
    sections = doc.get('sections', []) or []

    for sec in sections:
        heading = sec.get('heading')
        tables = sec.get('tables', []) or []
        for table_index, table in enumerate(tables):
            header = table.get('header') or []
            table_rows = table.get('rows') or []
            rows.append({
                'DOI': doi,
                'Section': heading,
                'Table_Index': table_index,
                'MD_Path': md_path,
                'PDF_Path': pdf_path,
                'Header': header,
                'Preview_Row': table_rows[0] if table_rows else [],
                'Rows': table_rows,
            })

df_tables = pd.DataFrame(rows)
print(f'Total extracted tables: {len(df_tables)}')
display(df_tables[['DOI', 'Section', 'Table_Index', 'MD_Path', 'PDF_Path', 'Header', 'Preview_Row']].head(50))

Total extracted tables: 609


,DOI,Section,Table_Index,MD_Path,PDF_Path,Header,Preview_Row
0,10.1186/s13601-019-0278-3,Results,0,paper_pipeline_data/md/PMC6706894.md,paper_pipeline_data/pdf/PMC6706894.pdf,"[Study, Type of study, Patients, Control group]",[I. Psychiatric comorbidity in CU patients (st...
1,10.1186/s13601-019-0278-3,Results,1,paper_pipeline_data/md/PMC6706894.md,paper_pipeline_data/pdf/PMC6706894.pdf,"[, CU patients, CU patients with psychiatric c...","[Juhlin [18], , ]"
2,10.1186/s13601-019-0278-3,Results,2,paper_pipeline_data/md/PMC6706894.md,paper_pipeline_data/pdf/PMC6706894.pdf,"[, Chronic urticaria patients, Control individ...","[All, With psychiatric comorbidity, All]"
3,10.1186/s13601-019-0278-3,Results,3,paper_pipeline_data/md/PMC6706894.md,paper_pipeline_data/pdf/PMC6706894.pdf,"[DSM-5 classification, PatientsN/all CU patien...","[Prevalence per study, Pooled prevalence (%), ..."
4,10.1002/clt2.12283,RESULTS,0,paper_pipeline_data/md/PMC10349543.md,paper_pipeline_data/pdf/PMC10349543.pdf,"[First author Year (Ref.), Sample size (T/C), ...","[Gerasimov 2010, 43/47, 25.6 ± 7.7/24.1 ± 6.3 ..."
5,10.1002/clt2.12283,RESULTS,1,paper_pipeline_data/md/PMC10349543.md,paper_pipeline_data/pdf/PMC10349543.pdf,"[Author, year, Randomization process, Deviatio...","[Gerasimov 2010, Low risk, Low risk, Low risk,..."
6,10.1002/clt2.12283,Study selection,0,paper_pipeline_data/md/PMC10349543.md,paper_pipeline_data/pdf/PMC10349543.pdf,"[First author Year (Ref.), Sample size (T/C), ...","[Gerasimov 2010, 43/47, 25.6 ± 7.7/24.1 ± 6.3 ..."
7,10.1002/clt2.12283,Risk of bias in the included studies,0,paper_pipeline_data/md/PMC10349543.md,paper_pipeline_data/pdf/PMC10349543.pdf,"[Author, year, Randomization process, Deviatio...","[Gerasimov 2010, Low risk, Low risk, Low risk,..."
8,10.3389/falgy.2024.1499406,Methods and analysis,0,paper_pipeline_data/md/PMC11694227.md,paper_pipeline_data/pdf/PMC11694227.pdf,"[No., Search terms]","[#1, MeSH terms: “rhinitis, allergic”]"
9,10.3389/falgy.2024.1499406,Databases and search strategies,0,paper_pipeline_data/md/PMC11694227.md,paper_pipeline_data/pdf/PMC11694227.pdf,"[No., Search terms]","[#1, MeSH terms: “rhinitis, allergic”]"


In [6]:
def show_table(index):
    if index < 0 or index >= len(df_tables):
        print('Index out of range')
        return

    record = df_tables.iloc[index]
    print('DOI:', record['DOI'])
    print('Section:', record['Section'])
    print('MD Path:', record['MD_Path'])
    print('PDF Path:', record['PDF_Path'])

    headers = record['Header'] or []
    table_rows = record['Rows'] or []
    if not headers or not table_rows:
        print('No table data available')
        return

    display(pd.DataFrame(table_rows, columns=headers))

if len(df_tables):
    show_table(7)
else:
    print('No tables to show')

DOI: 10.1002/clt2.12283
Section: Risk of bias in the included studies
MD Path: paper_pipeline_data/md/PMC10349543.md
PDF Path: paper_pipeline_data/pdf/PMC10349543.pdf


,"Author, year",Randomization process,Deviation from intended interventions,Missing outcome data,Measurement of the outcome,Selection of the reported result,Overall
0,Gerasimov 2010,Low risk,Low risk,Low risk,Some concerns,Some concerns,Some concerns
1,Wu 2011,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk
2,Han 2012,Low risk,Low risk,Low risk,Some concerns,Low risk,Some concerns
3,Wang 2015,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk
4,Navarro‐López 2017,Some concerns,Low risk,Low risk,Some concerns,Low risk,Some concerns
5,Plummer 2019,Low risk,Some concerns,Low risk,Low risk,Low risk,Some concerns
6,Jeong 2020,Some concerns,Some concerns,Low risk,Some concerns,Low risk,Some concerns
7,Ahn 2020,Some concerns,Low risk,Low risk,Some concerns,Low risk,Some concerns
8,Cukrowska 2021,Some concerns,Low risk,Low risk,Some concerns,Low risk,Some concerns


## Usage

- Use `df_tables` to scan the available tables.
- Call `show_table(i)` to render table `i`.
- Use the printed `MD Path` and `PDF Path` for quick source-file verification.